## Domain 5: Security Policies & Unified Governance
This notebook implements Row-Level Security (RLS), Column-Level Security (CLS), and fine-grained Access Control (RBAC) to protect sensitive ecommerce data.

In [0]:
%sql
USE CATALOG ecommerce_analytics_dev;
USE SCHEMA gold_layer;

## 1. Row-Level Security: Business Segmentation
We define a filter function to ensure data silos are respected. Marketing users only see 'top-of-funnel' events, while Finance only sees revenue-generating 'purchases'.

In [0]:
%sql
CREATE OR REPLACE FUNCTION gold_layer.rls_event_filter(event_type STRING)
RETURNS BOOLEAN
RETURN
  -- Safety check for your specific identity during the demo
  current_user() = 'ruchikamhetre25_gmail.com#ext#@ruchikamhetre25gmail.onmicrosoft.com' OR 
  is_account_group_member('admins') OR 
  is_account_group_member('ops_team') OR
  (is_account_group_member('finance_team') AND event_type = 'purchase') OR
  (is_account_group_member('marketing_team') AND event_type IN ('view','cart'));

## 2. Column-Level Security: Sensitive Data Masking
These functions protect PII (User IDs) and financial data (Price). Users without explicit 'Ops' or 'Finance' roles see NULLs or hashed values.

In [0]:
%sql
-- Masking for Financial Data
CREATE OR REPLACE FUNCTION gold_layer.mask_price(p DOUBLE)
RETURNS DOUBLE
RETURN
  CASE
    WHEN is_account_group_member('finance_team') OR 
         is_account_group_member('ops_team') OR 
         is_account_group_member('admins') THEN p
    ELSE NULL
  END;

-- Masking for PII (User Identification)
CREATE OR REPLACE FUNCTION gold_layer.mask_user(u STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('ops_team') OR 
         is_account_group_member('admins') THEN u
    ELSE sha1(u) -- Standard hashing for unauthorized users
  END;

## 3. Policy Enforcement
We bind the security functions to the physical `fact_sales` table. This ensures that the security logic is enforced regardless of which tool (SQL, Python, Power BI) accesses the data.

In [0]:
%sql
ALTER TABLE ecommerce_analytics_dev.gold_layer.fact_sales 
SET ROW FILTER gold_layer.rls_event_filter ON (event_type);

ALTER TABLE ecommerce_analytics_dev.gold_layer.fact_sales 
ALTER COLUMN price SET MASK gold_layer.mask_price;

ALTER TABLE ecommerce_analytics_dev.gold_layer.fact_sales 
ALTER COLUMN user_id SET MASK gold_layer.mask_user;

## 4. Role-Based Access Control (RBAC)
Finalizing the security model by granting minimum necessary privileges to specific business units.

In [0]:
%sql
-- 1. Grant Catalog Usage (Individual Statements)
GRANT USAGE ON CATALOG ecommerce_analytics_dev TO `finance_team`;
GRANT USAGE ON CATALOG ecommerce_analytics_dev TO `marketing_team`;
GRANT USAGE ON CATALOG ecommerce_analytics_dev TO `ops_team`;

-- 2. Grant Schema Usage
GRANT USAGE ON SCHEMA ecommerce_analytics_dev.gold_layer TO `finance_team`;
GRANT USAGE ON SCHEMA ecommerce_analytics_dev.gold_layer TO `marketing_team`;
GRANT USAGE ON SCHEMA ecommerce_analytics_dev.gold_layer TO `ops_team`;

-- 3. Grant Table Selection
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.fact_sales TO `finance_team`;
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.fact_sales TO `marketing_team`;
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.fact_sales TO `ops_team`;

-- 4. Grant Selection on KPI Aggregates
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.product_performance TO `finance_team`;
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.product_performance TO `marketing_team`;
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.product_performance TO `ops_team`;